# Practice run: our test portfolio

This notebook runs **every section of MyPortfolio** on our test portfolio for the competition, using the positions file exported from the broker on **October 24, 2026** (`OpenPosition_10_24_2026.csv`, in this folder).

| Section | What it tells us |
|---|---|
| MyStocks | What we own, what it's worth, and our gains and losses |
| MyRisks | Where the portfolio could get hurt, as a 0 to 100 risk score |
| MyOptimizer | The best mix of these same stocks for the starting rules |
| MyAnalysis | The final verdict on every holding, with reasons |

The results below were saved when the notebook was last run. **Run All** to refresh them with today's prices. The full verdict takes several minutes the first time.

> Everything here is a rule of thumb based on past data, not financial advice.

## Setup

In [1]:
# This notebook lives in practice/, so tell Python where the core/ folder is (one level up).
import sys
sys.path.insert(0, "..")

import warnings
warnings.simplefilter("ignore")           # each section's own notebook explains its warnings

import pandas as pd
from core import portfolio as pf, my_stocks as ms, my_risks as mr, my_optimizer as mo, verdict as vd

pd.set_option("display.max_colwidth", None)
print("Ready!")

Ready!


## 1. Load the broker's positions file

The broker's file has one row per holding: symbol, quantity, currency, today's price, and the average price we paid.

In [2]:
broker = pd.read_csv("OpenPosition_10_24_2026.csv", thousands=",")
broker[["Symbol", "Description", "Quantity", "Currency", "LastPrice", "PricePaid", "MarketValue", "ProfitLossPercentage"]]

,Symbol,Description,Quantity,Currency,LastPrice,PricePaid,MarketValue,ProfitLossPercentage
0,ABBV,Abbvie Inc,127,USD,285.64,274.570000,50807.114846,4.03
1,BDT,Bird Construction Inc.,566,CAD,69.10,72.517880,39110.600000,-4.71
2,FTNT,Fortinet Inc,61,USD,179.99,194.600000,15377.296919,-7.51
3,GLD,SPDR Gold Trust - SPDR Gold Shares ETF,247,USD,366.61,383.420000,126824.467787,-4.38
4,GOOG,Alphabet Inc - Ordinary Shares - Class C,100,USD,317.28,348.160000,44436.974790,-8.87
5,JNJ,Johnson & Johnson,135,USD,263.81,258.390000,49880.042017,2.10
6,JPM,JPMorgan Chase & Co.,105,USD,331.58,331.960000,48761.764706,-0.11
7,L,Loblaw Companies Limited,1012,CAD,68.67,63.804921,69494.040000,7.62
8,NVDA,NVIDIA Corp,247,USD,210.96,230.990000,72979.159664,-8.67
9,POW,Power Corporation of Canada Subordinate Voting Shares,566,CAD,92.42,88.580000,52309.720000,4.34


## 2. Turn it into a MyPortfolio portfolio

Two things need fixing so Yahoo Finance understands the symbols:

* **Canadian stocks need ".TO"** on the end (Toronto Stock Exchange). The broker writes `RY`, Yahoo wants `RY.TO`. We can tell they're Canadian because their currency is CAD.
* **Yahoo writes share classes with a dash.** The broker writes `QBR.B`, Yahoo wants `QBR-B.TO`.

The broker's file **doesn't include purchase dates**, so every holding is recorded as bought today. That's fine for gains (they use the price paid), but MyStocks' "S&P 500 twin" comparison can't be used.

In [3]:
def to_yahoo(symbol: str, currency: str) -> str:
    # "QBR.B" + CAD -> "QBR-B.TO";  "NVDA" + USD -> "NVDA"
    if currency == "CAD":
        return symbol.replace(".", "-") + ".TO"
    return symbol

test_portfolio = pf.empty_portfolio()
for _, row in broker.iterrows():
    pf.add_holding(test_portfolio, to_yahoo(row["Symbol"], row["Currency"]), row["Quantity"], row["PricePaid"])

pf.holdings_table(test_portfolio)

,ticker,shares,buy_price,buy_date
0,ABBV,127.0,274.570000,2026-10-24
1,BDT.TO,566.0,72.517880,2026-10-24
2,FTNT,61.0,194.600000,2026-10-24
3,GLD,247.0,383.420000,2026-10-24
4,GOOG,100.0,348.160000,2026-10-24
5,JNJ,135.0,258.390000,2026-10-24
6,JPM,105.0,331.960000,2026-10-24
7,L.TO,1012.0,63.804921,2026-10-24
8,NVDA,247.0,230.990000,2026-10-24
9,POW.TO,566.0,88.580000,2026-10-24


## 3. MyStocks: what we own and how it's doing

Canadian and US stocks are kept in **separate groups**, each in its own currency. Only the pie chart and the combined total convert, using today's exchange rate.

In [4]:
positions = ms.positions_table(test_portfolio)
positions.style.format(
    {"shares": "{:,.0f}", "avg_cost": "{:,.2f}", "cost": "{:,.0f}", "price": "{:,.2f}", "value": "{:,.0f}",
     "gain": "{:+,.0f}", "gain_pct": "{:+.1f}%", "day_change": "{:+,.0f}", "day_change_pct": "{:+.2f}%",
     "weight_pct": "{:.1f}%", "pe_ratio": "{:.1f}", "dividend_yield_pct": "{:.2f}%"},
    na_rep="n/a",
)

,ticker,name,currency,sector,shares,avg_cost,cost,price,value,gain,gain_pct,day_change,day_change_pct,weight_pct,pe_ratio,dividend_yield_pct
0,ZLB.TO,BMO LOW VOLATILITY CAD EQUITY E,CAD,n/a,"2,491",58.60,"145,973",61.25,"152,574","+6,601",+4.5%,"-1,769",-1.15%,15.9%,n/a,n/a
1,QBR-B.TO,QUEBECOR INC CLASS B SV,CAD,Communication Services,"1,733",64.04,"110,981",65.39,"113,321","+2,340",+2.1%,"-1,681",-1.46%,11.8%,16.0,2.26%
2,L.TO,LOBLAW CO,CAD,Consumer Defensive,"1,012",63.80,"64,571",68.67,"69,494","+4,923",+7.6%,"-2,014",-2.82%,7.3%,30.9,0.82%
3,RY.TO,ROYAL BANK OF CANADA,CAD,Financial Services,193,272.53,"52,598",281.77,"54,382","+1,783",+3.4%,-195,-0.36%,5.7%,17.8,2.33%
4,POW.TO,"POWER CORPORATION OF CANADA, SV",CAD,Financial Services,566,88.58,"50,136",92.42,"52,310","+2,173",+4.3%,"-1,149",-2.15%,5.5%,22.8,2.65%
5,SU.TO,SUNCOR ENERGY INC.,CAD,Energy,451,103.21,"46,548",111.18,"50,142","+3,594",+7.7%,"+1,258",+2.57%,5.2%,14.8,2.19%
6,BDT.TO,BIRD CONSTRUCTION INC.,CAD,Industrials,566,72.52,"41,045",69.10,"39,111","-1,935",-4.7%,+362,+0.93%,4.1%,64.6,1.23%
7,GLD,SPDR Gold Shares,USD,n/a,247,383.42,"94,705",366.61,"90,553","-4,152",-4.4%,-590,-0.65%,13.2%,n/a,0.00%
8,NVDA,NVIDIA Corporation,USD,Technology,247,230.99,"57,055",210.96,"52,107","-4,947",-8.7%,-309,-0.59%,7.6%,26.7,0.13%
9,ABBV,AbbVie Inc.,USD,Healthcare,127,274.57,"34,870",285.64,"36,276","+1,406",+4.0%,+57,+0.16%,5.3%,80.7,2.39%


In [5]:
summary = ms.portfolio_summary(test_portfolio)
for currency, g in summary["groups"].items():
    print(f"--- {currency} stocks ({g['stocks']} holdings) ---")
    print(f"Worth now: {g['value']:>14,.0f} {currency}")
    print(f"We paid:   {g['cost']:>14,.0f} {currency}")
    print(f"Gain:      {g['gain']:>+14,.0f} {currency}  ({g['gain_pct']:+.1f}%)")
    print(f"Today:     {g['day_change']:>+14,.0f} {currency}  ({g['day_change_pct']:+.2f}%)")
    print()
combined = summary["combined"]
print(f"Everything together: {combined['value']:,.0f} {combined['currency']}  (exchange rate used: {combined['fx_rates']})")
print(f"The broker's own total: {broker['MarketValue'].sum():,.0f} CAD")

--- CAD stocks (7 holdings) ---
Worth now:        531,333 CAD
We paid:          511,852 CAD
Gain:             +19,481 CAD  (+3.8%)
Today:             -5,187 CAD  (-0.97%)

--- USD stocks (8 holdings) ---
Worth now:        304,095 USD
We paid:          318,377 USD
Gain:             -14,282 USD  (-4.5%)
Today:             -1,928 USD  (-0.63%)

Everything together: 957,188 CAD  (exchange rate used: {'USD': 1.4004000425338745})
The broker's own total: 957,236 CAD


In [6]:
ms.allocation_chart(positions).show()

## 4. MyRisks: where could it get hurt?

Eight areas, each low, medium or high, rolled into a **risk score from 0 to 100 (higher is riskier)**. Measured on the past three years of prices.

**Note:** the sector check counts the two funds, gold (GLD) and the low-volatility Canadian fund (ZLB), together as "Fund or other", so the sector picture is rough.

In [7]:
risks = mr.portfolio_risks(test_portfolio)
print(f"Risk score: {risks['score']:.0f} / 100, {risks['level'].upper()}")
print(f"Measured from {risks['period']['start']} to {risks['period']['end']}.")
print()
for number, risk in enumerate(risks["top_risks"], start=1):
    print(f"{number}. {risk}")
risks["areas"]

Risk score: 0 / 100, LOW
Measured from 2023-10-25 to 2026-10-23.

1. NVDA is 8% of your money but 21% of your risk. Trimming it would calm the portfolio down the most.


,level,reading
area,,
Concentration,low,Biggest: ZLB.TO 16%. Top 3: 41%. Acts like 11.2 equal stocks.
Sector,low,"29% in Fund or other, across 8 sectors."
Currency,low,"CAD 56%, USD 44%. A 10% move in exchange rates would change your total by about $42,585 CAD."
Moving together,low,Average correlation 0.13. Closest pair: ZLB.TO and L.TO (0.56).
Who brings risk,low,"NVDA: 8% of your money, 21% of your risk. Yearly swings 9.7%."
Market risk,low,"Beta 0.49. If the S&P 500 fell 20%, you'd expect about -10% ($93,076 CAD)."
Worst drop,low,"-7.8% at its worst, on 2025-04-08. Worst single stock: BDT.TO -43%."
Bad day,low,"About 1 day in 20, you could lose 0.9% ($8,522 CAD) or more. On those days the average loss is 1.3%."


In [8]:
mr.sector_chart(risks).show()
mr.correlation_chart(risks).show()

In [9]:
mr.risk_share_chart(risks).show()
mr.drawdown_chart(risks).show()

## 5. MyOptimizer: the best mix of these same stocks

The starting rules: highest Sharpe ratio, worst drop no bigger than 25%, at least 8% a year, at most 30% in any one stock. It's based on the past three years, so stocks that did especially well recently look extra good.

In [10]:
optimized = mo.optimize_portfolio(test_portfolio)
print(optimized["message"])
optimized["comparison"].style.format(
    {"yearly return %": "{:.1f}%", "yearly swings %": "{:.1f}%", "Sharpe ratio": "{:.2f}",
     "worst drop %": "{:.1f}%", "$10,000 became": "${:,.0f}"}
)

Out of 20,000 mixes, 20,000 follow your rules. The best one for the highest Sharpe ratio is shown.


,yearly return %,yearly swings %,Sharpe ratio,worst drop %,"$10,000 became"
Your current mix,39.5%,9.7%,3.67,-7.8%,"$27,606"
Best mix,45.7%,9.4%,4.42,-7.6%,"$31,517"


In [11]:
# Sorted from the biggest suggested cut to the biggest suggested addition.
optimized["weights"].sort_values("change (points)")

,current %,best %,change (points)
ticker,,,
ZLB.TO,15.9,0.0,-15.9
GLD,13.2,5.5,-7.7
JPM,5.1,0.0,-5.1
GOOG,4.6,0.0,-4.6
ABBV,5.3,1.8,-3.6
NVDA,7.6,5.2,-2.4
PWR,1.8,0.0,-1.8
QBR-B.TO,11.8,13.4,1.6
FTNT,1.6,3.5,1.9


In [12]:
mo.weights_chart(optimized).show()
mo.cloud_chart(optimized).show()

In [13]:
# Check the best mix's risk before switching to it.
best_mix_risks = mr.mix_risks(optimized["best_weights"], value=risks["value"])
print(f"Our current mix: risk score {risks['score']:.0f} ({risks['level']})")
print(f"The best mix:    risk score {best_mix_risks['score']:.0f} ({best_mix_risks['level']})")

Our current mix: risk score 0 (low)
The best mix:    risk score 0 (low)


## 6. MyAnalysis: the final verdict on every holding

Runs all six parts of MyAnalysis (business health, rivals, DCF value, economy, price chart, analysts) on every holding, **most worrying first**. Funds like GLD and ZLB have no financial statements, so only some parts can run for them.

**This takes several minutes the first time**, because the rivals part downloads data for every holding's competitors.

In [14]:
review = vd.portfolio_review(test_portfolio)
review["table"].style.format({"score": "{:.0f}"}, na_rep="n/a")

,name,score,verdict,momentum,top concern,flag
ticker,,,,,,
GLD,SPDR Gold Shares,38,look closer,weak,Trend (price chart): downtrend.,True
RY.TO,ROYAL BANK OF CANADA,55,hold,mixed,Looks overvalued: DCF value 140.24 is 50% below today's price.,False
JNJ,Johnson & Johnson,56,hold,mixed,Looks overvalued: DCF value 181.16 is 31% below today's price.,False
GOOG,Alphabet Inc.,57,hold,mixed,Looks overvalued: DCF value 105.96 is 67% below today's price.,False
FTNT,"Fortinet, Inc.",58,hold,strong,Looks overvalued: DCF value 89.03 is 51% below today's price.,False
POW.TO,"POWER CORPORATION OF CANADA, SV",59,hold,strong,Looks overvalued: DCF value 66.17 is 28% below today's price.,False
ABBV,AbbVie Inc.,61,hold,strong,"Economy headwind from interest rates: This sector usually isn't very sensitive to interest rates. It carries a lot of debt (20.2x equity), so rising rates hurt more. Its P/E is high (81x), so rising rates hurt more: its value leans on far-future profits.",False
PWR,"Quanta Services, Inc.",61,hold,mixed,Looks overvalued: DCF value 455.60 is 17% below today's price.,False
JPM,JP Morgan Chase & Co.,63,hold,strong,Looks overvalued: DCF value 109.21 is 67% below today's price.,False


In [15]:
# The reasons behind each verdict, most worrying first.
for ticker in review["table"].index:
    result = review["results"][ticker]
    print(f"{ticker}: {result['score']:.0f} / 100, {result['verdict']} (momentum {result['momentum']['reading']})")
    for reason in result["strengths"][:2]:
        print("   +", reason)
    for reason in result["concerns"][:2]:
        print("   -", reason)
    print()

GLD: 38 / 100, look closer (momentum weak)
   + Reward to risk (price chart): 3.0 to 1, good.
   - Trend (price chart): downtrend.
   - Moving averages (price chart): below both.

RY.TO: 55 / 100, hold (momentum mixed)
   + Profitable: net margin 32.0%.
   + Competitive advantage is strong against rivals (83/100).
   - Looks overvalued: DCF value 140.24 is 50% below today's price.
   - Margins not shrinking check failed: net margin trend: worsening.

JNJ: 56 / 100, hold (momentum mixed)
   + Profitable: net margin 21.5%.
   + Trend (price chart): uptrend.
   - Looks overvalued: DCF value 181.16 is 31% below today's price.
   - Forecast changes (90 days) (analysts): -7.0%.

GOOG: 57 / 100, hold (momentum mixed)
   + Average rating (analysts): 1.87, buy.
   + Profitable: net margin 54.8%.
   - Looks overvalued: DCF value 105.96 is 67% below today's price.
   - Moving averages (price chart): below both.

FTNT: 58 / 100, hold (momentum strong)
   + Profitable: net margin 28.2%.
   + Trend 

In [16]:
# The six part scores for the holding that needs the closest look.
vd.verdict_chart(review["results"][review["table"].index[0]]).show()